# Historical screenshot sequence analysis

## What you will build

**Recipe 99 · Level 4 (Advanced) · Noul + images + long context.** Build a bounded analyzer for
a fictional UI test session that lasted 3.5 hours. It compares eight scheduled screenshots
with the complete synthetic log history to identify the earliest **retained visible transition**
associated with a reported chart-collapse defect. Python owns chronology, candidate generation,
request limits and final selection. The model never returns a timestamp or invents a candidate.

A [review](../../docs/glossary.md#review) is a deferral to a person; it is not an accepted answer.
Missing captures, uncertain earlier evidence, stale records and contradictions lead there.
An associated log does not prove a cause. A first visible after-frame does not locate the exact
onset between samples, and even complete scheduled sampling can miss short-lived changes.
Every UI image and log is authored synthetic material, and every action in this notebook is simulated.

Sources:

- S02: [Primitives and Noul](https://docs.typesafe.ai/primitives)
- S03: [Confidence](https://docs.typesafe.ai/confidence)
- S09: [Images](https://docs.typesafe.ai/images)
- S11: [Models and long context](https://docs.typesafe.ai/models)
- S12: [Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## Setup and run mode

The default [replay](../../docs/glossary.md#replay-key) looks up a stored answer using the
entire request and image bytes. A miss raises an error. The exact same questions can be sent
live with an explicit model pin and the experimental image and long-context flags.
The flag enables a capability; this small fixture does not benchmark a full 262K-token window.
The README describes deliberate recording with `record(..., triples, ...)`.

The fixture sample has 12 validation sessions, 12 held-out test sessions and two unscored demos.
A cache prevents illustration or threshold selection from issuing the same request twice.

In [ ]:
import json
from dataclasses import asdict

import matplotlib.pyplot as plt
from PIL import Image

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, run_header
from jev_cookbook.vision import run_record

helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
scored = [row for row in examples if row.split != "demo"]
backend = get_backend(
    fixtures=responses_path(),
    model="jev-1.14.0",
    images=True,
    long_context=True,
)
runner = helpers.SessionRunner(backend)
run_header(99, "Historical screenshot sequence analysis", backend=backend, n_examples=len(scored))
check = " — synthetic pipeline check, not a Jev result" if backend.mode == "synthetic" else ""
selection = " — validation selection step"
reportable = backend.mode != "live"
apply_style()
results = {}
actions, reviews = ActionLog(), ReviewQueue()

## The state

The screenshot schedule is 0 through 12,600 seconds in 1,800-second increments. Input rows are
intentionally shuffled; Python sorts explicit numeric timestamps and rejects duplicates,
nonfinite times and oversized requests. Every adjacent pair is a candidate, including earlier
benign changes. Logs are also sorted by event time; emission time remains separate so delayed
or stale events cannot masquerade as current evidence. The log history includes heartbeats,
refresh operations and unrelated errors instead of only the apparent failure window.

The request contains pixels, the defect report, timestamps, operation identifiers and log text.
Gold annotations, split names and drawing plans are excluded. Image references identify content;
they do not describe what the image depicts. These are original code-drawn screenshots of a
fictional Telemetry Lab UI, not captures of a real application.

In [ ]:
demo = next(row for row in examples if row.id == "d01")
state, demo_questions, demo_assets = helpers.request(demo.fields)
print(
    {
        "session_seconds": state["frames"][-1]["timestamp_seconds"],
        "screenshots": len(state["frames"]),
        "log_rows": len(state["logs"]),
        "serialized_text_characters": len(json.dumps(state)),
        "input_tokens": None,
        "candidate_intervals": [
            (row["id"], row["start_seconds"], row["end_seconds"]) for row in state["candidates"]
        ],
    }
)
print("First and last chronological log:", state["logs"][0], state["logs"][-1])
fig, axes = plt.subplots(2, 4, figsize=(14, 5.8))
for axis, frame in zip(axes.flat, helpers.ordered_frames(demo.fields), strict=True):
    with Image.open(frame["path"]) as image:
        axis.imshow(image)
    axis.set_title(f"{frame['timestamp_seconds']} seconds")
    axis.axis("off")
fig.suptitle("Original synthetic screenshots in chronological order")
fig.tight_layout()
fig

## The questions

A `Noul` proposition returns a probability of true ([S02](https://docs.typesafe.ai/primitives)).
The first proposition checks whether the symptom was already visible at the first capture.
Each candidate gets two separate propositions: a new visible collapse, and a relevant current
log association. Their judgments are independent, so Python batches all 15 into one request
with the full history. No question asks the model to count time, choose the earliest item,
prove causality or take an action. Logs are evidence, including instruction-like strings;
they do not authorize changes to the fixed question set.

For incomplete sampling there are fewer candidates, still within eight images and fifteen
questions. Such answers can aid review, but cannot authorize an earliest-transition report.

In [ ]:
print("Questions in this one request:", len(demo_questions))
for name in ("initial", "c4_visible", "c4_associated"):
    print(name, demo_questions[name].to_dict())

## One answer up close

This is the typed answer for the pair from 5,400 to 7,200 seconds. The selected after-frame
shows the chart as a thin strip. The earlier frame shows a tall panel. The log below is
associated evidence; it does not explain why the layout changed.

In [ ]:
results[demo.id] = runner.decide(demo.fields)
for name in ("initial", "c3_visible", "c4_visible", "c4_associated"):
    print(name, results[demo.id].answers[name].to_dict())
print("Associated log:", next(row for row in state["logs"] if row["operation_id"] == "refresh-4"))
print("Accounting:", run_record(results[demo.id], mode=backend.mode))

## Python's part

The [confidence gate](../../docs/glossary.md#confidence-gate) requires enough certainty before
accepting a judgment. For `Noul` the recipe uses `abs(2*p - 1)`, a derived certainty rather than
an API confidence field ([S03](https://docs.typesafe.ai/confidence)). A tied or malformed answer
always defers. The demo cutoff below is illustrative; validation chooses the reporting cutoff.

Python first checks the complete scheduled sample grid and a confidently negative initial
presence. It walks candidates in chronological order. Every earlier visual proposition must
be confidently false before a later one can be accepted. A positive visual result then needs
a matching current-session operation, a fresh failure log, no contradictory success record,
and a confident semantic association. Any failure stops the walk in review. The reported
interval merely brackets two retained samples; it is not an inferred exact onset interval.

A negative final answer means no target transition in the retained screenshots. All requests
are bounded and there are no retries at the recipe layer. The backend separately limits HTTP
attempts. Reports below enter an `ActionLog` with `executed=False`; review entries wait in a
`ReviewQueue` and change nothing.

In [ ]:
demo_outcome = helpers.route(demo.fields, results[demo.id].answers, 0.6)
actions.record(
    "draft_transition_report", asdict(demo_outcome), rule="complete chronological prefix"
)
missing_demo = next(row for row in examples if row.id == "d02")
results[missing_demo.id] = runner.decide(missing_demo.fields)
missing_outcome = helpers.route(missing_demo.fields, results[missing_demo.id].answers, 0.6)
reviews.submit({"session": missing_demo.fields["session_id"]}, missing_outcome.reason)
print("Simulated report:", actions.to_dicts())
print("Review queue:", reviews.to_dicts())
print(
    "Request limits:",
    {
        "logical_calls": helpers.MAX_REQUESTS,
        "frames": helpers.MAX_FRAMES,
        "logs": helpers.MAX_LOGS,
        "text_characters": helpers.MAX_CONTEXT_CHARACTERS,
    },
)

## Evaluation

A [gold label](../../docs/glossary.md#gold-label) is an authored reference answer in a separate
file. Here it names an earliest after-frame time, `none`, or `review` when evidence cannot
support a complete answer. An accepted answer matching a review label is an error.
[Coverage](../../docs/glossary.md#coverage) is the accepted fraction of sessions; a review is
never coverage. Event [accuracy](../../docs/glossary.md#accuracy) is the correct fraction of
accepted earliest-event answers, and [risk](../../docs/glossary.md#risk) is one minus that
accuracy. We also print correct accepted events divided by all sessions to expose abstention.

The candidate family is fixed at certainty cutoffs 0.4, 0.6, 0.8 and 0.95. On validation only,
maximize **correct accepted minus four times wrong accepted**; break ties by more accepted
sessions and then the lower cutoff. Structural checks stay fixed. The selected policy is
frozen before reading test answers. This is a small synthetic policy exercise, not calibration
of a real model. Live exploration withholds aggregate metrics until answers are recorded.

The deterministic baseline computes mean absolute RGB pixel change over the full image and
pairs a change of at least 0.02 with an `error` keyword inside the same interval. This fixed
baseline is intentionally simple but real: it reads pixels and logs, never gold labels.

In [ ]:
validation = [row for row in examples if row.split == "validation"]
for row in validation:
    results[row.id] = runner.decide(row.fields)
if reportable:
    frozen_threshold, policy_rows = helpers.select_policy(
        validation,
        results,
        labels,
        objective="correct_minus_four_errors",
    )
    for row in policy_rows:
        print(f"N={len(validation)} {row}{selection}{check}")
    print(f"Frozen certainty={frozen_threshold}, N={len(validation)}{selection}{check}")
else:
    frozen_threshold = 0.6
    print("Live exploration uses the documented cutoff; record before selecting or reporting.")

In [ ]:
test = [row for row in examples if row.split == "test"]
for row in test:
    results[row.id] = runner.decide(row.fields)
test_outcomes = [
    helpers.route(row.fields, results[row.id].answers, frozen_threshold) for row in test
]
baseline_outcomes = [helpers.baseline(row.fields) for row in test]
for row, outcome in zip(test, test_outcomes, strict=True):
    if not outcome.accepted:
        reviews.submit({"session": row.fields["session_id"]}, outcome.reason)
    elif outcome.event != helpers.NONE:
        actions.record("draft_transition_report", asdict(outcome), rule="frozen policy")
if reportable:
    test_gold = [labels[row.id] for row in test]
    policy_metrics = helpers.summarize(test_outcomes, test_gold)
    baseline_metrics = helpers.summarize(baseline_outcomes, test_gold)
    print(f"Held-out chronological policy: {policy_metrics}{check}")
    print(f"Held-out pixel/log baseline: {baseline_metrics}{check}")
    for row, outcome, baseline_outcome in zip(test, test_outcomes, baseline_outcomes, strict=True):
        print(
            f"{row.id}: gold={labels[row.id]['event']}, policy={outcome.event}, "
            f"accepted={outcome.accepted}, baseline={baseline_outcome.event}; {outcome.reason}{check}"
        )
else:
    print("Unrecorded live exploration: held-out aggregates withheld.")
print("Logical requests made:", runner.calls)

### Compare answered events and deferrals

These plotted values come from the same once-evaluated test outcomes. The baseline answers
every session, including those with evidence that needs review. The policy can leave an
answer unresolved, so compare its accepted-event accuracy alongside coverage, not by itself.

In [ ]:
if reportable:
    plotted = {
        "chronological policy": [
            policy_metrics["coverage"],
            policy_metrics["event_accuracy_accepted"],
        ],
        "pixel/log baseline": [
            baseline_metrics["coverage"],
            baseline_metrics["event_accuracy_accepted"],
        ],
    }
    print(f"Test N={len(test)} plotted values {plotted}{check}")
    fig, axis = plt.subplots(figsize=(7, 3.5))
    x = [0, 1]
    axis.bar(
        [n - 0.18 for n in x],
        plotted["chronological policy"],
        width=0.36,
        color="#F386A1",
        label="Chronological policy",
    )
    axis.bar(
        [n + 0.18 for n in x],
        plotted["pixel/log baseline"],
        width=0.36,
        color="#E551BA",
        label="Pixel/log baseline",
    )
    axis.set_xticks(x, ["Coverage", "Event accuracy among accepted"])
    axis.set_ylim(0, 1.18)
    axis.set_title(f"Held-out sessions, N={len(test)}{check}", fontsize=10)
    axis.legend(loc="upper left", fontsize=9)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

### A confident accepted mistake

The held-out error below survives the frozen policy. The authored replay incorrectly rejects
the real early transition and accepts a later frame where the collapsed chart was already
present. Timestamp arithmetic and provenance checks cannot repair a wrong visual judgment.
This is why the request controls are not an accuracy guarantee.

In [ ]:
if reportable:
    wrong = [
        (row, outcome)
        for row, outcome in zip(test, test_outcomes, strict=True)
        if outcome.accepted and outcome.event != labels[row.id]["event"]
    ]
    print(f"Accepted mistakes={len(wrong)} of test N={len(test)}{check}")
    for row, outcome in wrong:
        print(f"{row.id}: predicted {outcome.event}; reference {labels[row.id]['event']}{check}")
        print(labels[row.id]["rationale"])
        print("Early visual answer:", results[row.id].answers["c3_visible"].to_dict())
        print("Later visual answer:", results[row.id].answers["c5_visible"].to_dict())
    row = next(row for row in test if row.id == "t10")
    ordered = helpers.ordered_frames(row.fields)
    fig, axes = plt.subplots(1, 3, figsize=(10, 2.8))
    for axis, frame in zip(axes, [ordered[2], ordered[3], ordered[5]], strict=True):
        with Image.open(frame["path"]) as image:
            axis.imshow(image)
        axis.set_title(f"{frame['timestamp_seconds']} seconds")
        axis.axis("off")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

## What was and was not measured

The cell states mode, model and capture date where applicable, and N for the validation and
test splits. This committed run replays authored synthetic answers and is **not measured
live**. It checks the pipeline, including one confidently wrong held-out result; it does not
measure Jev quality. Unknown usage, inference latency and cost remain `None`. Replay wall-clock
time is not inference latency. The full log history covers 3.5 simulated hours, while eight
scheduled stills only sample that history. Enabling long context does not measure token use.

A recorded replay reports only its returned model, capture dates and named sample. The README
shows how to collect the same requests into an all-recorded responses file with the supplied
recorder. Any new real dataset needs an untouched test set. An unrecorded live notebook is
exploration and supplies no reportable held-out result.

In [ ]:
if backend.mode == "synthetic":
    print("Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(scored)} scored; validation {len(validation)}, test {len(test)}, demos 2")
    print(
        f"Mode=synthetic; N={len(scored)} scored sessions; validation N={len(validation)}, "
        f"test N={len(test)}; demo N=2; not measured live.{check}"
    )
    print(
        {
            "model": None,
            "capture_date": None,
            "input_tokens": None,
            "output_tokens": None,
            "inference_latency_seconds": None,
            "cost_usd": None,
        }
    )
elif backend.mode == "recorded":
    print(
        f"Mode=recorded; model={backend.model}; capture dates={backend.recorded_dates}; "
        f"N={len(scored)} scored; validation N={len(validation)}, test N={len(test)}"
    )
    print(
        "Per-request accounting:",
        [run_record(result, mode=backend.mode) for result in results.values()],
    )
else:
    print(
        f"Mode=live exploration; returned model={backend.model}; N={len(scored)} scored inputs; "
        f"validation N={len(validation)}, test N={len(test)}; aggregates withheld"
    )
    print(
        "Per-request accounting:",
        [run_record(result, mode=backend.mode) for result in results.values()],
    )
print("Limits of inference: earliest retained evidence only; association does not establish cause.")

## Next steps

Use [evidence conflict diagnosis](../47-evidence-conflict-diagnosis/) to deepen conflicting-log
handling and [abstention and review queues](../38-abstention-and-review-queues/) to explore the
cost of deferral. For real screenshot histories, capture more densely, preserve trustworthy
session and event times, and label a new held-out set before comparing models. A separate
experiment could vary capture cadence while keeping the same underlying sessions; it would
need to distinguish sampling misses from judgment errors.